### importinf files and setup

In [1]:
import os
import sys
import numpy as np
import pandas as pd

project_root = r"C:\Users\Arafat\Desktop\IMA Project"
if project_root not in sys.path:
    sys.path.append(project_root)

from src import config
from src import data_loading
from src import features
from src import pipelines
from src.utils import set_random_seed

config.make_all_dirs()
set_random_seed(42)

print("Available pipelines:", list(pipelines.PIPELINES.keys()))


[utils] Random seed set to 42
Available pipelines: ['RawPixels', 'GrayHist', 'ColorHists', 'Texture_LBP_GLCM', 'EdgesAndTexture', 'Frequency_FFT_DCT', 'Morphology_GrayHist', 'Combo_BestGuess']


# Cell 2 – load split CSV and split DataFrames

In [2]:
# here I load the split info
df_splits = data_loading.load_splits_csv()
df_train, df_val, df_test = data_loading.get_split_dataframes(df_splits)

print("Train shape:", df_train.shape)
print("Val shape:", df_val.shape)
print("Test shape:", df_test.shape)

print("Train class distribution:")
print(df_train["label"].value_counts())


[data_loading] Loading split CSV from C:\Users\Arafat\Desktop\IMA Project\data\processed\splits\split_info.csv
[data_loading] Loaded DataFrame shape: (12444, 3)
[data_loading] Train size: 7466
[data_loading] Val size: 2489
[data_loading] Test size: 2489
Train shape: (7466, 3)
Val shape: (2489, 3)
Test shape: (2489, 3)
Train class distribution:
label
NEUTROPHIL    1873
EOSINOPHIL    1872
LYMPHOCYTE    1862
MONOCYTE      1859
Name: count, dtype: int64


# Cell 3 – label encoding helper

In [3]:
from sklearn.preprocessing import LabelEncoder

# I encode string labels -> integer class ids
label_encoder = LabelEncoder()
label_encoder.fit(df_splits["label"])

train_labels_encoded = label_encoder.transform(df_train["label"])
val_labels_encoded = label_encoder.transform(df_val["label"])
test_labels_encoded = label_encoder.transform(df_test["label"])

print("Label classes (in order):", list(label_encoder.classes_))

# I also save mapping to a csv, useful for report
label_map_df = pd.DataFrame({
    "class_id": np.arange(len(label_encoder.classes_)),
    "class_name": label_encoder.classes_,
})
print("Label mapping:")
print(label_map_df)

mapping_path = os.path.join(config.PROCESSED_DATA_DIR, "label_mapping.csv")
label_map_df.to_csv(mapping_path, index=False)
print("Saved label mapping to:", mapping_path)


Label classes (in order): ['EOSINOPHIL', 'LYMPHOCYTE', 'MONOCYTE', 'NEUTROPHIL']
Label mapping:
   class_id  class_name
0         0  EOSINOPHIL
1         1  LYMPHOCYTE
2         2    MONOCYTE
3         3  NEUTROPHIL
Saved label mapping to: C:\Users\Arafat\Desktop\IMA Project\data\processed\label_mapping.csv


# Cell 4 – choose which pipelines to compute now

In [4]:
# here I choose which pipelines to actually run now
# later I can come back and run the others

selected_pipeline_names = [
    "RawPixels",
    "GrayHist",
    "ColorHists",
    "Texture_LBP_GLCM",
    "EdgesAndTexture",
    "Frequency_FFT_DCT",
    "Morphology_GrayHist",
    "Combo_BestGuess",
]

print("Pipelines I will extract features for:", selected_pipeline_names)


Pipelines I will extract features for: ['RawPixels', 'GrayHist', 'ColorHists', 'Texture_LBP_GLCM', 'EdgesAndTexture', 'Frequency_FFT_DCT', 'Morphology_GrayHist', 'Combo_BestGuess']


# Cell 5 – feature extraction loop (this is the main heavy cell)

In [5]:
# In this cell I loop over each selected pipeline and build features for
# train, val, and test sets, then save them as .npz files.

from src.utils import SimpleTimer

features_root = config.FEATURES_DIR
if not os.path.exists(features_root):
    os.makedirs(features_root, exist_ok=True)

for pipe_name in selected_pipeline_names:
    pipe_info = pipelines.PIPELINES[pipe_name]
    feature_name_list = pipe_info["feature_names"]
    print("\n========================================")
    print("Pipeline:", pipe_name)
    print("Description:", pipe_info["description"])
    print("Feature names:", feature_name_list)

    # I create a subfolder just for this pipeline
    pipe_folder = os.path.join(features_root, pipe_name)
    if not os.path.exists(pipe_folder):
        os.makedirs(pipe_folder, exist_ok=True)

    # Build feature matrices
    with SimpleTimer(message=f"Building features for pipeline {pipe_name} (train)"):
        train_paths = df_train["filepath"].tolist()
        feat_train = features.build_feature_matrix(train_paths, feature_name_list)

    with SimpleTimer(message=f"Building features for pipeline {pipe_name} (val)"):
        val_paths = df_val["filepath"].tolist()
        feat_val = features.build_feature_matrix(val_paths, feature_name_list)

    with SimpleTimer(message=f"Building features for pipeline {pipe_name} (test)"):
        test_paths = df_test["filepath"].tolist()
        feat_test = features.build_feature_matrix(test_paths, feature_name_list)

    print("Train feature matrix shape:", feat_train.shape)
    print("Val feature matrix shape:", feat_val.shape)
    print("Test feature matrix shape:", feat_test.shape)

    # save as compressed npz
    save_path = os.path.join(pipe_folder, f"{pipe_name}_features.npz")
    np.savez_compressed(
        save_path,
        feat_train=feat_train,
        feat_val=feat_val,
        feat_test=feat_test,
        y_train=train_labels_encoded,
        y_val=val_labels_encoded,
        y_test=test_labels_encoded,
    )
    print("Saved features to:", save_path)



Pipeline: RawPixels
Description: Simple baseline using raw grayscale pixel values.
Feature names: ['raw_pixels']
[timer] Starting: Building features for pipeline RawPixels (train)
[features] Building feature matrix...
[features] Number of images: 7466
[features] Feature types: ['raw_pixels']
[features] Feature dimension per image: 16384
[features] Processed 20/7466 images
[features] Processed 40/7466 images
[features] Processed 60/7466 images
[features] Processed 80/7466 images
[features] Processed 100/7466 images
[features] Processed 120/7466 images
[features] Processed 140/7466 images
[features] Processed 160/7466 images
[features] Processed 180/7466 images
[features] Processed 200/7466 images
[features] Processed 220/7466 images
[features] Processed 240/7466 images
[features] Processed 260/7466 images
[features] Processed 280/7466 images
[features] Processed 300/7466 images
[features] Processed 320/7466 images
[features] Processed 340/7466 images
[features] Processed 360/7466 image

C:\Users\Arafat\AppData\Local\Programs\Python\Python312\Lib\site-packages\skimage\feature\texture.py:360: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(


[features] Processed 20/7466 images
[features] Processed 40/7466 images
[features] Processed 60/7466 images
[features] Processed 80/7466 images
[features] Processed 100/7466 images
[features] Processed 120/7466 images
[features] Processed 140/7466 images
[features] Processed 160/7466 images
[features] Processed 180/7466 images
[features] Processed 200/7466 images
[features] Processed 220/7466 images
[features] Processed 240/7466 images
[features] Processed 260/7466 images
[features] Processed 280/7466 images
[features] Processed 300/7466 images
[features] Processed 320/7466 images
[features] Processed 340/7466 images
[features] Processed 360/7466 images
[features] Processed 380/7466 images
[features] Processed 400/7466 images
[features] Processed 420/7466 images
[features] Processed 440/7466 images
[features] Processed 460/7466 images
[features] Processed 480/7466 images
[features] Processed 500/7466 images
[features] Processed 520/7466 images
[features] Processed 540/7466 images
[feat